# **---------New Phase-------------------------------------**

In [26]:
%pip uninstall transformers==5.18.0

%pip install --no-cache-dir \
    transformers==4.46.3 \
    tokenizers==0.20.3

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Would remove:
    /usr/local/bin/transformers-cli
    /usr/local/lib/python3.13/dist-packages/transformers-4.46.3.dist-info/*
    /usr/local/lib/python3.13/dist-packages/transformers/*
Proceed (Y/n)? y
  Successfully uninstalled transformers-4.46.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 188.0 MB/s eta 0:00:00


In [1]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

assert transformers.__version__ == "4.46.3"

Transformers: 4.46.3
PyTorch: 2.11.0+cu130


In [2]:
pip install sacrebleu

In [3]:
import random
import json
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import sacrebleu

from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)

import transformers
print("Transformers:", transformers.__version__)

assert transformers.__version__ == "4.46.3"

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 4.46.3
GPU: NVIDIA A100-SXM4-80GB


In [4]:
BASE = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/DataSet"
)


TRAIN_AR = BASE / "opus.ar-en-train.ar"
TRAIN_EN = BASE / "opus.ar-en-train.en"

DEV_AR = BASE / "opus.ar-en-dev.ar"
DEV_EN = BASE / "opus.ar-en-dev.en"

TEST_AR = BASE / "opus.ar-en-test.ar"
TEST_EN = BASE / "opus.ar-en-test.en"

for p in [
    TRAIN_AR, TRAIN_EN,
    DEV_AR, DEV_EN,
    TEST_AR, TEST_EN,
]:
    print(p.name, "PASS" if p.exists() else "MISSING")

opus.ar-en-train.ar PASS
opus.ar-en-train.en PASS
opus.ar-en-dev.ar PASS
opus.ar-en-dev.en PASS
opus.ar-en-test.ar PASS
opus.ar-en-test.en PASS


In [5]:
def read_lines(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return [line.strip() for line in f]

train_ar = read_lines(TRAIN_AR)
train_en = read_lines(TRAIN_EN)

dev_ar = read_lines(DEV_AR)
dev_en = read_lines(DEV_EN)

test_ar = read_lines(TEST_AR)
test_en = read_lines(TEST_EN)

assert len(train_ar) == len(train_en)
assert len(dev_ar) == len(dev_en)
assert len(test_ar) == len(test_en)

print("Train:", len(train_ar))
print("Dev:", len(dev_ar))
print("Test:", len(test_ar))

Train: 1000000
Dev: 2000
Test: 2000


In [6]:
from pathlib import Path

LOCAL_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "mt5_en_ar_full_clean/checkpoints"
)

DRIVE_BACKUP_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "mt5_en_ar_full_clean/backups"
)

FINAL_RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "mt5_en_ar_full_clean"
)

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
FINAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
dataset = DatasetDict({
    "train": Dataset.from_dict({
        "source": train_en,
        "target": train_ar,
    }),
    "validation": Dataset.from_dict({
        "source": dev_en,
        "target": dev_ar,
    }),
    "test": Dataset.from_dict({
        "source": test_en,
        "target": test_ar,
    }),
})

In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(LOCAL_OUTPUT_DIR),

    max_steps=31250,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=32,
    gradient_checkpointing=True,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,

    evaluation_strategy="steps",
    eval_steps=2500,

    save_strategy="steps",
    save_steps=2500,
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=100,

    predict_with_generate=False,

    bf16=True,
    fp16=False,

    max_grad_norm=1.0,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    seed=42,
    data_seed=42,
)

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
MODEL_NAME = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

model.gradient_checkpointing_enable()
model.config.use_cache = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model:", MODEL_NAME)
print("Device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Model: google/mt5-base
Device: cuda


In [10]:
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["source", "target"],
    num_proc=1,
    desc="Tokenizing",
)

print(tokenized_dataset)

Tokenizing (num_proc=1):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Streaming output truncated to the last 5000 lines.
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a3999e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a398b80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a3fd760>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.p

Tokenizing (num_proc=1):   0%|          | 0/2000 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7a3437526570>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py", line 698, in convert_to_tensors
    def convert_to_tensors(
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a3437526570>
Traceback (most recent call last):
  File "/usr/lib/python3.13/contextlib.py", line 136, in __enter__
    def __enter__(self):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a404f40>
Traceback (most recent call last):
  File "/usr/lib/python3.13/contextlib.py", line 136, in __enter__
    def __enter__(self):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a405300>
Traceback (most recent call last):
  File "/usr/lib/python3.13/contextlib.py", line 136, in __enter__
    def __enter__(se

Tokenizing (num_proc=1):   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})


In [12]:
sample_source = dataset["train"][0]["source"]
sample_target = dataset["train"][0]["target"]

src_tokens = tokenizer(
    sample_source,
    max_length=256,
    truncation=True
)

tgt_tokens = tokenizer(
    text_target=sample_target,
    max_length=256,
    truncation=True
)

print("Raw source:")
print(sample_source)

print("\nDecoded source:")
print(
    tokenizer.decode(
        src_tokens["input_ids"],
        skip_special_tokens=False
    )
)

print("\nRaw target:")
print(sample_target)

print("\nDecoded target:")
print(
    tokenizer.decode(
        tgt_tokens["input_ids"],
        skip_special_tokens=False
    )
)

Raw source:
And this?

Decoded source:
And this?</s>

Raw target:
و هذه؟

Decoded target:
و هذه؟</s>


In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

In [23]:

from transformers import TrainerCallback

class VerifiedCheckpointBackupCallback(TrainerCallback):

    def on_save(self, args, state, control, **kwargs):

        step = state.global_step

        local_ckpt = (
            Path(args.output_dir) /
            f"checkpoint-{step}"
        )

        print("\n" + "=" * 70)
        print(f"VERIFYING LOCAL CHECKPOINT {step}")
        print("=" * 70)

        required = [
            "model.safetensors",
            "optimizer.pt",
            "scheduler.pt",
            "trainer_state.json",
            "training_args.bin",
            "rng_state.pth",
        ]

        missing = [
            name
            for name in required
            if not (local_ckpt / name).exists()
        ]

        if missing:
            print("LOCAL CHECKPOINT INCOMPLETE")
            print("Missing:", missing)
            return control

        for name in required:
            p = local_ckpt / name

            print(
                f"{name:25s}",
                f"{p.stat().st_size / 1024**2:.2f} MB"
            )

        # Create local tar archive
        local_archive = Path(
            f"/content/checkpoint-{step}.tar"
        )

        if local_archive.exists():
            local_archive.unlink()

        print("\nCreating local archive...")

        with tarfile.open(local_archive, "w") as tar:
            tar.add(
                local_ckpt,
                arcname=f"checkpoint-{step}"
            )

        local_hash = sha256_file(local_archive)

        print(
            "Archive size:",
            f"{local_archive.stat().st_size / 1024**3:.2f} GB"
        )
        print("Local SHA256:", local_hash)

        # Temporary Drive copy
        drive_tmp = (
            DRIVE_BACKUP_DIR /
            f"checkpoint-{step}.tar.copying"
        )

        drive_final = (
            DRIVE_BACKUP_DIR /
            f"checkpoint-{step}.tar"
        )

        if drive_tmp.exists():
            drive_tmp.unlink()

        print("Copying archive to Drive...")

        shutil.copy2(
            local_archive,
            drive_tmp
        )

        os.sync()
        time.sleep(10)

        if not drive_tmp.exists():
            print("DRIVE COPY FAILED")
            return control

        drive_hash = sha256_file(drive_tmp)

        print("Drive SHA256:", drive_hash)
        print("SHA256 MATCH:", local_hash == drive_hash)

        if local_hash != drive_hash:
            print("DRIVE BACKUP FAILED")
            return control

        if drive_final.exists():
            drive_final.unlink()

        os.replace(
            drive_tmp,
            drive_final
        )

        marker = (
            DRIVE_BACKUP_DIR /
            f"checkpoint-{step}.verified"
        )

        marker.write_text(
            f"step={step}\n"
            f"sha256={drive_hash}\n",
            encoding="utf-8"
        )

        print(
            f"\nCHECKPOINT {step}: "
            "VERIFIED AND BACKED UP TO DRIVE"
        )

        return control

In [15]:
import os
import shutil
import time
import tarfile
import hashlib
from pathlib import Path
import tarfile

In [16]:
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

In [24]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[
        VerifiedCheckpointBackupCallback()
    ],
)

/tmp/ipykernel_105358/581336605.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
max_steps is given, it will override any value given in num_train_epochs


In [18]:
train_result = trainer.train()

Exception ignored in: <_io.BytesIO object at 0x7a342a3504a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a39b380>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a352110>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7a342a3e2930>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-package

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [19]:
from pathlib import Path

LOCAL_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "mt5_en_ar_full_clean/checkpoints"
)

restore_dir = LOCAL_OUTPUT_DIR / "checkpoint-2500"

for name in [
    "model.safetensors",
    "optimizer.pt",
    "scheduler.pt",
    "trainer_state.json",
    "training_args.bin",
    "rng_state.pth",
]:
    p = restore_dir / name
    print(
        f"{name:25s}",
        "PASS" if p.exists() else "MISSING",
        f"{p.stat().st_size / 1024**2:.2f} MB"
        if p.exists() else ""
    )

model.safetensors         PASS 2221.72 MB
optimizer.pt              PASS 4443.60 MB
scheduler.pt              PASS 0.00 MB
trainer_state.json        PASS 0.00 MB
training_args.bin         PASS 0.01 MB
rng_state.pth             PASS 0.01 MB


In [20]:
from pathlib import Path

candidates = list(
    Path("/content").glob("checkpoint-2500*.tar")
)

print("Archives found:")

for p in candidates:
    print(
        p,
        f"{p.stat().st_size / 1024**3:.2f} GB"
    )

Archives found:
/content/checkpoint-2500.tar 6.51 GB


In [21]:
import shutil
import os
import time
from pathlib import Path

DRIVE_BACKUP_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "mt5_en_ar_full_clean/backups"
)

DRIVE_BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

local_archive = candidates[0]

drive_temp = (
    DRIVE_BACKUP_DIR /
    "checkpoint-2500.tar.copying"
)

drive_final = (
    DRIVE_BACKUP_DIR /
    "checkpoint-2500.tar"
)

marker = (
    DRIVE_BACKUP_DIR /
    "checkpoint-2500.verified"
)

print("Hashing local archive...")
local_hash = sha256_file(local_archive)

print("Local SHA256:", local_hash)

print("Copying to Drive...")
shutil.copy2(
    local_archive,
    drive_temp
)

os.sync()
time.sleep(10)

print("Hashing Drive copy...")
drive_hash = sha256_file(drive_temp)

print("Drive SHA256:", drive_hash)
print("MATCH:", local_hash == drive_hash)

assert local_hash == drive_hash

os.replace(
    drive_temp,
    drive_final
)

with open(
    marker,
    "w",
    encoding="utf-8"
) as f:
    f.write(
        f"step=2500\n"
        f"sha256={local_hash}\n"
        f"size={drive_final.stat().st_size}\n"
    )

print()
print("CHECKPOINT 2500 BACKUP COMPLETE")
print("Archive:", drive_final)
print(
    "Size:",
    drive_final.stat().st_size / 1024**3,
    "GB"
)

Hashing local archive...
Local SHA256: cd522c69600a1496f0e73b2a08619f8985af7e603bcfd662890a24aa5fe50caa
Copying to Drive...
Hashing Drive copy...
Drive SHA256: cd522c69600a1496f0e73b2a08619f8985af7e603bcfd662890a24aa5fe50caa
MATCH: True

CHECKPOINT 2500 BACKUP COMPLETE
Archive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/mt5_en_ar_full_clean/backups/checkpoint-2500.tar
Size: 6.513156890869141 GB


In [22]:
import tarfile
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

In [26]:
import os

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [27]:
print(
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD =",
    os.environ.get("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD")
)

TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD = 1


In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=str(restore_dir)
)

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
/usr/local/lib/python3.13/dist-packages/transformers/trainer.py:3354: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), map_location=map_location)
/usr/local/lib/python3.13/dist-packages/transformers/trainer.py:3033: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint_rng_state = torch.load(rng_file)


Step,Training Loss,Validation Loss
22500,2.551300,2.115837
25000,2.550200,2.099860
27500,2.543700,2.091987
30000,2.509600,2.087221



VERIFYING LOCAL CHECKPOINT 22500
model.safetensors         2221.72 MB
optimizer.pt              4443.60 MB
scheduler.pt              0.00 MB
trainer_state.json        0.04 MB
training_args.bin         0.01 MB
rng_state.pth             0.01 MB

Creating local archive...
Archive size: 6.51 GB
Local SHA256: 6363a403aab1e9cebadfc2c28321cdc748417177b0e93f07d0f0b8bf9ac4f6a9
Copying archive to Drive...
Drive SHA256: 6363a403aab1e9cebadfc2c28321cdc748417177b0e93f07d0f0b8bf9ac4f6a9
SHA256 MATCH: True

CHECKPOINT 22500: VERIFIED AND BACKED UP TO DRIVE

VERIFYING LOCAL CHECKPOINT 25000
model.safetensors         2221.72 MB
optimizer.pt              4443.60 MB
scheduler.pt              0.00 MB
trainer_state.json        0.04 MB
training_args.bin         0.01 MB
rng_state.pth             0.01 MB

Creating local archive...
Archive size: 6.51 GB
Local SHA256: e6f5087ce3c8a00b44e6d280708606e0aa5b24ad55c9d20c7c57ddc767d529c9
Copying archive to Drive...
Drive SHA256: e6f5087ce3c8a00b44e6d280708606e0aa5b2

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
